In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.spatial.distance as spd
import pickle
from sklearn.metrics import confusion_matrix
from numpy.linalg import pinv

In [3]:
with open("A50731_Q001_data.p", "rb") as data_file:
    D = pickle.load(data_file)

In [4]:
D

{'x': array([3.64572321, 4.60388183, 5.88585463, 4.48877315, 5.68929889,
        7.45491232, 4.9961246 , 7.10500346, 3.47371961, 0.54239595,
        3.87545563, 5.95139864, 2.44509227, 3.29151717, 5.50357532,
        7.99902639, 5.42534464, 5.39878968, 7.49698228, 6.50646302,
        1.83415161, 3.77952359, 4.10921636, 5.47529105, 6.1200911 ,
        2.13598949, 3.05115412, 2.33061108, 7.95440443, 5.46251042,
        2.29129043, 7.96003409, 6.70968297, 6.50055931, 8.43604976,
        2.86674187, 7.92473197, 1.67294756, 4.3988604 , 7.99042847,
        8.37945566, 3.99389014, 2.29907491, 3.2886074 , 7.87382964,
        6.35116069, 0.49005948, 0.99896806, 5.8733815 , 8.56261683,
        0.7232218 , 2.48303848, 3.66439159, 5.79914414, 1.57247981,
        1.19206155, 8.88690498, 7.67989839, 5.92633263, 3.10949718,
        6.12397683, 4.18509758, 7.48463356, 3.2164806 , 8.37027432,
        7.74489201, 7.08082456, 4.76957472, 4.25468088, 3.72985921,
        6.96192551, 1.67844662, 3.71761858,

In [5]:
D.keys()

dict_keys(['x', 'y', 'folds'])

In [14]:
# Separar as variáveis
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures


x = np.array(D['x']).reshape(-1, 1)
y = np.array(D['y'])
folds = np.array(D['folds'])

# Criar os conjuntos de treino e teste
train_indices = folds == 0
test_indices = folds == 1

x_train, y_train = x[train_indices], y[train_indices]
x_test, y_test = x[test_indices], y[test_indices]

# Regressão polinomial de 4ª ordem
poly = PolynomialFeatures(degree=4)
x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

model = LinearRegression()
model.fit(x_train_poly, y_train)

# Previsões
y_train_pred = model.predict(x_train_poly)
y_test_pred = model.predict(x_test_poly)

# Métricas
mse_test = mean_squared_error(y_test, y_test_pred)
r2_train = r2_score(y_train, y_train_pred)

np.round(mse_test, 2), np.round(r2_train, 2)

(0.05, 0.93)

In [18]:
# Ajustar o modelo para um polinômio de 3ª ordem
poly = PolynomialFeatures(degree=3)
x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

model = LinearRegression()
model.fit(x_train_poly, y_train)

# Previsões no conjunto de treino
y_train_pred = model.predict(x_train_poly)

# Métricas
r2_train = r2_score(y_train, y_train_pred)
coefficients = model.intercept_, *model.coef_

np.round(r2_train, 2), np.round(coefficients[0], 2)  # Coeficiente de determinação e w0


(0.79, 2.91)

In [20]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
import numpy as np

# Carregar o conjunto de dados
data = load_diabetes()
X, y = data.data, data.target

# Dividir os dados em treino (primeiras 312 amostras) e teste (restantes)
X_train, y_train = X[:312], y[:312]
X_test, y_test = X[312:], y[312:]

# Gerar termos polinomiais de 3ª ordem
poly = PolynomialFeatures(degree=3)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Ajustar o modelo Lasso
lasso = Lasso(random_state=42, alpha=0.01)
lasso.fit(X_train_poly, y_train)

# Previsões no conjunto de teste
y_test_pred = lasso.predict(X_test_poly)

# Calcular métricas
mse_test = mean_squared_error(y_test, y_test_pred)

# Contar coeficientes com valor absoluto maior que 246 (excluindo w0)
coef_abs = np.abs(lasso.coef_)
num_large_coefs = np.sum(coef_abs > 246)

np.round(mse_test, 2), np.round(num_large_coefs, 2)


(2757.17, 12)

In [22]:
# Gerar termos polinomiais de 2ª ordem
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Ajustar o modelo de regressão linear
model = LinearRegression()
model.fit(X_train_poly, y_train)

# Previsões no conjunto de teste
y_test_pred = model.predict(X_test_poly)

# Métricas
mse_test = mean_squared_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

np.round(mse_test, 2), np.round(r2_test, 2)


(3369.77, 0.41)

In [24]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# Carregar os dados do ficheiro
with open("A50731_Q003_data.p", "rb") as file:
    data = pickle.load(file)

In [25]:
data.keys()

dict_keys(['y', 'X'])

In [57]:
# Separar as variáveis
X = data['X']  # matriz de dados
y = data['y']  # classes (-1 e +1)

# Vetor de pesos fornecido
w = np.array([0.00, 0.73, 0.69])  # w0, w1, w2

X = X.T

# Adicionar x0 = 1 para todas as amostras (termo bias)
X_with_bias = np.hstack((np.ones((X.shape[0], 1)), X))

# Verificando as formas antes da multiplicação
print("Forma de X_with_bias:", X_with_bias.shape)  # Deve ser (2575, 3)
print("Forma de w:", w.shape)  # Deve ser (3,)

# Calcular as predições lineares (verificando a transposição de X_with_bias se necessário)
y_hat = np.dot(X_with_bias, w)  # Alternativa com np.dot para garantir produto escalar

# Converter predições em classes (+1 para y_hat >= 0, -1 para y_hat < 0)
y_pred = np.where(y_hat >= 0, 1, -1)




y_pred = np.where(y_pred == -1, 0, 1)  # Converte -1 para 0 e 1 mantém como 1

print(np.unique(y))  # Isso irá mostrar as classes presentes em y
print(np.unique(y_pred))  # Esperado: [0, 1]
# Calcular métricas
precision = precision_score(y, y_pred, average='binary')
recall = recall_score(y, y_pred, average='binary')
cm = confusion_matrix(y, y_pred)


# Extrair valores da matriz de confusão
TN, FP, FN, TP = cm.ravel()

# Número de acertos na classe positiva (TP)
num_correct_positive = TP

# Taxa de falsos alarmes
false_alarm_rate = FP / (FP + TN)

np.round(precision,3), np.round(num_correct_positive,3), np.round(false_alarm_rate, 3), np.round(recall,3)


Forma de X_with_bias: (2574, 3)
Forma de w: (3,)
[0. 1.]
[0 1]


(0.888, 1687, 0.301, 0.902)

In [65]:
mse = np.mean((y - y_pred) ** 2)
print("Erro quadrático médio (MSE):", np.round(mse, 3))
print("Acertos classe 0:", TN)


Erro quadrático médio (MSE): 0.153
Acertos classe 0: 492


In [ ]:
#1 a- [0,1,0,0]

In [ ]:
#1 b- [0,0,0,1]

In [ ]:
#2 a- [0,0,0,1]

In [ ]:
#2 b- [0,0,0,1]

In [58]:
#3 a- [1, 0, 0, 0]

In [ ]:
#3 b- [0,0,0,1]